# Geosteering AI v2.40 — Validação JAX GPU

**Template**: `validate_jax_gpu_v240.ipynb` | **Sprint**: v2.40 (I2.2 MCP colab-bridge)

Valida os **109 testes JAX** do projeto em GPU T4 (Colab Pro+), respondendo à
pergunta arquitetural: **"É possível automatizar testes JAX GPU via MCP colab-bridge?"** — Sim, via este template (caminho D3-a do plano).

**Pré-requisito**: Runtime GPU T4 ou superior (Runtime → Change runtime type → GPU).

**Saída**: `/content/drive/.../paridade_jax_gpu_T4.json` com:
- Status de cada teste (PASS/FAIL/SKIPPED)
- Paridade JAX GPU vs Numba CPU por modelo canônico (gate <1e-10)
- Latência por teste (samples/sec)

## Cell 1 — Configuração

In [ ]:
GIT_TAG = "v2.40"
OUTPUT_DRIVE_DIR = "/content/drive/MyDrive/Geosteering_AI/v2.40/colab_runs/"
PARIDADE_TOL = 1e-10  # gate JAX vs Numba

print(f"✓ Configuração: GIT_TAG={GIT_TAG}, tolerância={PARIDADE_TOL}")

## Cell 2 — Setup do Ambiente + Validação GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone --depth 1 --branch {GIT_TAG} https://github.com/daniel-guitarplayer-8/geosteering-ai.git
%cd geosteering-ai
!pip install -q -e ".[all]"
!pip install -q pytest-json-report

# Validar JAX detectou GPU
import jax
devices = jax.devices()
gpu_devices = [d for d in devices if d.platform == "gpu"]
assert len(gpu_devices) > 0, f"Nenhuma GPU JAX detectada! Devices: {devices}"
print(f"✓ JAX devices: {devices}")
print(f"✓ GPU: {gpu_devices[0]}")
# Sprint v2.40.1 (Security #6 + hardening F1): pin commit hash imutável
# Tag pode ser movida; hash é imutável → rastreabilidade total do run.
# try/except graceful: se repo foi baixado como ZIP (sem .git/) ou
# instalado via pip git+... (sem clone), usa fallback baseado em tag.
import os, subprocess
_REPO_DIR = "/content/geosteering-ai"
try:
    GIT_COMMIT = subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        cwd=_REPO_DIR if os.path.isdir(_REPO_DIR + "/.git") else ".",
    ).decode().strip()
    print(f"✓ Pinned commit: {GIT_COMMIT[:12]}")
except (subprocess.CalledProcessError, FileNotFoundError) as _e:
    GIT_COMMIT = f"unknown-from-tag-{GIT_TAG}"
    print(f"⚠ git rev-parse falhou ({_e}); fallback: {GIT_COMMIT}")


## Cell 3 — Executar Testes JAX em GPU (109 testes)

Com `JAX_PLATFORMS=gpu` setado, o conftest detecta GPU e os 109 testes
marcados com `@pytest.mark.gpu` (Sprint v2.40 C06) são EXECUTADOS
(em vez de SKIPPED como em CPU).

In [ ]:
import os
os.environ["JAX_PLATFORMS"] = "gpu"

# Roda 109 testes JAX com pytest + json report
!python -m pytest tests/test_simulation_jax_*.py \
    -m gpu \
    -v \
    --json-report \
    --json-report-file=/tmp/jax_gpu_report.json \
    2>&1 | tail -30

## Cell 4 — Paridade JAX GPU vs Numba CPU (7 modelos canônicos)

Para cada modelo canônico do simulador, executa em JAX GPU e Numba CPU,
compara `max(|diff|)` por componente do tensor H 3×3.
Gate: < 1e-10 → PASS, ≥ 1e-10 → XFAIL (paridade degradada, investigação futura).

In [ ]:
import numpy as np
from geosteering_ai.simulation import simulate_multi
from geosteering_ai.simulation.config import SimulationConfig

# 7 modelos canônicos (homogêneo, contraste, multi-camada, anisotropia, etc.)
MODELOS = [
    {"nome": "homogeneo_1", "rho_h": [10.0], "rho_v": [10.0], "esp": [], "z_top": 0.0},
    {"nome": "contraste_1_100", "rho_h": [1.0, 100.0], "rho_v": [1.0, 100.0], "esp": [10.0], "z_top": -5.0},
    {"nome": "tri_camada", "rho_h": [1.0, 50.0, 5.0], "rho_v": [1.0, 50.0, 5.0], "esp": [5.0, 5.0], "z_top": -5.0},
    {"nome": "aniso_2_1", "rho_h": [2.0, 20.0], "rho_v": [4.0, 10.0], "esp": [10.0], "z_top": -5.0},
    {"nome": "high_contrast", "rho_h": [0.5, 500.0], "rho_v": [0.5, 500.0], "esp": [10.0], "z_top": -5.0},
    {"nome": "sandstone", "rho_h": [10.0, 50.0, 10.0], "rho_v": [10.0, 50.0, 10.0], "esp": [3.0, 3.0], "z_top": -3.0},
    {"nome": "shale_oil", "rho_h": [2.0, 200.0, 2.0], "rho_v": [2.0, 200.0, 2.0], "esp": [10.0, 10.0], "z_top": -10.0},
]

results = []
for m in MODELOS:
    # CPU Numba
    cfg_cpu = SimulationConfig(backend="numba")
    H_cpu = simulate_multi(cfg=cfg_cpu, rho_h=np.array(m["rho_h"]), rho_v=np.array(m["rho_v"]),
                            esp=np.array(m["esp"]), positions_z=np.linspace(-2, 2, 20),
                            tr_spacings_m=[1.0], dip_degs=[0.0], frequencies_hz=[20000.0])
    # JAX GPU
    cfg_jax = SimulationConfig(backend="jax")
    H_jax = simulate_multi(cfg=cfg_jax, rho_h=np.array(m["rho_h"]), rho_v=np.array(m["rho_v"]),
                            esp=np.array(m["esp"]), positions_z=np.linspace(-2, 2, 20),
                            tr_spacings_m=[1.0], dip_degs=[0.0], frequencies_hz=[20000.0])
    max_diff = float(np.max(np.abs(np.asarray(H_cpu) - np.asarray(H_jax))))
    paridade_ok = max_diff < PARIDADE_TOL
    results.append({
        "modelo": m["nome"],
        "max_abs_diff": max_diff,
        "paridade_ok": paridade_ok,
        "tolerancia": PARIDADE_TOL,
    })
    status = "✓" if paridade_ok else "⚠"
    print(f"{status} {m['nome']}: max_diff={max_diff:.3e} (tol={PARIDADE_TOL:.0e})")

n_pass = sum(1 for r in results if r["paridade_ok"])
print(f"\nParidade: {n_pass}/{len(results)} modelos PASS")

## Cell 5 — Exportar JSON para Drive

In [ ]:
import os
import json
import datetime

# Carregar relatório pytest
with open("/tmp/jax_gpu_report.json") as f:
    pytest_report = json.load(f)

ts = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
os.makedirs(OUTPUT_DRIVE_DIR, exist_ok=True)
out_path = os.path.join(OUTPUT_DRIVE_DIR, f"paridade_jax_gpu_T4_{ts}.json")

output = {
    "template": "validate_jax_gpu_v240",
    "git_tag": GIT_TAG,
    "git_commit": GIT_COMMIT,  # v2.40.1 Security #6
    "runtime": "colab_pro_plus_T4",
    "timestamp_utc": ts,
    "pytest_summary": pytest_report["summary"],
    "pytest_duration_sec": pytest_report.get("duration", 0.0),
    "paridade_jax_vs_numba": {
        "tolerance": PARIDADE_TOL,
        "modelos": results,
        "n_pass": sum(1 for r in results if r["paridade_ok"]),
        "n_total": len(results),
    },
    "exit_code": 0 if pytest_report["summary"].get("failed", 0) == 0 else 1,
}

with open(out_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"✓ Resultados exportados para: {out_path}")

## Cell 6 — Resumo Final

In [ ]:
summary = pytest_report["summary"]
print("=" * 60)
print("SPRINT v2.40 — VALIDAÇÃO JAX GPU CONCLUÍDA")
print("=" * 60)
print(f"Pytest total:    {summary.get('total', 0)}")
print(f"  PASSED:        {summary.get('passed', 0)}")
print(f"  FAILED:        {summary.get('failed', 0)}")
print(f"  SKIPPED:       {summary.get('skipped', 0)}")
print(f"  ERROR:         {summary.get('error', 0)}")
print(f"")
print(f"Paridade JAX GPU vs Numba CPU (tolerância {PARIDADE_TOL:.0e}):")
for r in results:
    status = "✓" if r['paridade_ok'] else "⚠"
    print(f"  {status} {r['modelo']:20s} max_diff={r['max_abs_diff']:.3e}")
print("=" * 60)
print(f"JSON exportado: {out_path}")